# Data Preprocessing & Feature Engineering
## Objectives

This notebook prepares the maternal health dataset for machine learning by:

- Removing unnecessary features
- Encoding categorical variables
- Creating a processed dataset
- Standardizing numerical variables
- Saving both processed and standardized datasets

In [1]:
# Import Libraries

import polars as pl
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [2]:
# Display Settings

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

polars.config.Config

In [4]:
# Load Dataset

df = pl.read_csv("../data/raw/maternal_health_dataset.csv")

df.head()

id,age_years,gravidity,parity,gestational_age_weeks,bmi_pre_pregnancy,systolic_bp_mmhg,diastolic_bp_mmhg,hemoglobin_gdl,anemia_status,fasting_glucose_mgdl,proteinuria,hiv_status,anc_visits,delivery_mode,primary_complication,pregnancy_outcome,risk_level
i64,i64,i64,i64,f64,f64,i64,i64,f64,str,i64,i64,i64,i64,str,str,str,str
1,23,3,2,33.7,24.7,121,80,10.9,"""mild""",75,0,0,6,"""vaginal""","""none""","""live_birth""","""low"""
2,28,5,4,31.0,15.4,103,68,10.5,"""mild""",74,0,0,4,"""vaginal""","""none""","""live_birth""","""low"""
3,20,3,1,24.5,29.9,117,65,12.5,"""none""",85,0,1,5,"""vaginal""","""none""","""live_birth""","""high"""
4,31,2,1,33.5,23.0,131,99,12.9,"""none""",87,2,0,4,"""vaginal""","""preeclampsia""","""live_birth""","""moderate"""
5,32,9,8,22.3,25.4,159,90,12.3,"""none""",77,4,0,4,"""vaginal""","""preeclampsia""","""live_birth""","""moderate"""


In [5]:
df = df.drop([
    "id",
    "anemia_status",
    "primary_complication"
])

df.columns

['age_years',
 'gravidity',
 'parity',
 'gestational_age_weeks',
 'bmi_pre_pregnancy',
 'systolic_bp_mmhg',
 'diastolic_bp_mmhg',
 'hemoglobin_gdl',
 'fasting_glucose_mgdl',
 'proteinuria',
 'hiv_status',
 'anc_visits',
 'delivery_mode',
 'pregnancy_outcome',
 'risk_level']

## Observation

Three non-predictive or redundant features were removed to improve model generalisation and reduce the likelihood of data leakage.

In [6]:
df.dtypes

[Int64,
 Int64,
 Int64,
 Float64,
 Float64,
 Int64,
 Int64,
 Float64,
 Int64,
 Int64,
 Int64,
 Int64,
 String,
 String,
 String]

## Observation

The dataset contains both numerical and categorical variables. Categorical variables will be encoded before model training.

In [7]:
categorical_columns = [
    "delivery_mode",
    "pregnancy_outcome",
    "risk_level"
]

encoders = {}

for col in categorical_columns:
    encoder = LabelEncoder()

    df = df.with_columns(
        pl.Series(
            col,
            encoder.fit_transform(df[col])
        )
    )

    encoders[col] = encoder

df.head()

age_years,gravidity,parity,gestational_age_weeks,bmi_pre_pregnancy,systolic_bp_mmhg,diastolic_bp_mmhg,hemoglobin_gdl,fasting_glucose_mgdl,proteinuria,hiv_status,anc_visits,delivery_mode,pregnancy_outcome,risk_level
i64,i64,i64,f64,f64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64
23,3,2,33.7,24.7,121,80,10.9,75,0,0,6,1,0,1
28,5,4,31.0,15.4,103,68,10.5,74,0,0,4,1,0,1
20,3,1,24.5,29.9,117,65,12.5,85,0,1,5,1,0,0
31,2,1,33.5,23.0,131,99,12.9,87,2,0,4,1,0,2
32,9,8,22.3,25.4,159,90,12.3,77,4,0,4,1,0,2


## Observation

Categorical variables were converted into numerical labels to make them suitable for machine learning algorithms.

In [8]:
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

df.write_csv(
    processed_path / "maternal_health_processed.csv"
)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


## Observation

The processed dataset was saved before feature scaling. This version will be used for tree-based algorithms such as Decision Tree and Random Forest.

In [9]:
numerical_columns = [
    "age_years",
    "gravidity",
    "parity",
    "gestational_age_weeks",
    "bmi_pre_pregnancy",
    "systolic_bp_mmhg",
    "diastolic_bp_mmhg",
    "hemoglobin_gdl",
    "fasting_glucose_mgdl",
    "proteinuria",
    "hiv_status",
    "anc_visits"
]

scaler = StandardScaler()

scaled_values = scaler.fit_transform(
    df.select(numerical_columns).to_numpy()
)

In [10]:
scaled_df = df.with_columns([
    pl.Series(
        col,
        scaled_values[:, i]
    )
    for i, col in enumerate(numerical_columns)
])

scaled_df.head()

age_years,gravidity,parity,gestational_age_weeks,bmi_pre_pregnancy,systolic_bp_mmhg,diastolic_bp_mmhg,hemoglobin_gdl,fasting_glucose_mgdl,proteinuria,hiv_status,anc_visits,delivery_mode,pregnancy_outcome,risk_level
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64
-0.552348,-0.078263,-0.031894,0.877562,0.231666,0.299944,0.663932,0.280773,-0.821646,-0.438478,-0.335121,0.938708,1,0,1
0.353279,0.816167,0.872485,0.503981,-1.936531,-0.830373,-0.424774,0.084025,-0.90439,-0.438478,-0.335121,0.046343,1,0,1
-1.095725,-0.078263,-0.484084,-0.395378,1.443991,0.048763,-0.69695,1.067767,0.005798,-0.438478,2.984,0.492526,1,0,0
0.896656,-0.525477,-0.484084,0.849889,-0.164671,0.927898,2.387715,1.264515,0.171286,1.711804,-0.335121,0.046343,1,0,2
1.077781,2.605026,2.681244,-0.699777,0.394864,2.68617,1.571186,0.969393,-0.656157,3.862086,-0.335121,0.046343,1,0,2


## Observation

The numerical variables were standardized while the encoded categorical variables remained unchanged.

In [11]:
scaled_df.write_csv(
    processed_path / "maternal_health_standardized.csv"
)

print("Standardized dataset saved successfully.")

Standardized dataset saved successfully.


## Observation

The standardized dataset has been saved and will be used as the input for Logistic Regression.

In [12]:
print("Processed Dataset Shape")

print(df.shape)

print()

print("Standardized Dataset Shape")

print(scaled_df.shape)

Processed Dataset Shape
(30000, 15)

Standardized Dataset Shape
(30000, 15)


# Summary

The preprocessing stage has successfully prepared the dataset for machine learning.

## Completed tasks:

- Removed unnecessary features.
- Encoded categorical variables.
- Saved the processed dataset.
- Standardized numerical variables.
- Saved the standardized dataset.

These outputs provide two versions of the dataset:

## Dataset	Purpose
maternal_health_processed.csv	Decision Tree & Random Forest
maternal_health_standardized.csv	Logistic Regression

The next stage is Model Development, where the processed datasets will be split into training and testing sets, followed by baseline modelling using Logistic Regression and performance comparison with Decision Tree and Random Forest.